In [25]:
import numpy as np
import pandas as pd
import re

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [ ]:
# df = pd.read_csv('data/cleaned.csv')
# X_train, X_test = train_test_split(df, test_size=0.2, stratify=df['Modelo'], random_state=42)
# # Save X_test to a CSV file inside the data folder
# X_test.to_csv('data/cleaned_test.csv', index=True)
# print(f"X_test dataset with {X_test.shape[0]} rows and {X_test.shape[1]} columns saved to data/cleaned_test.csv")
# X_train.to_csv('data/cleaned_train.csv', index=True)
# print(f"X_train dataset with {X_train.shape[0]} rows and {X_train.shape[1]} columns saved to data/cleaned_train.csv")

X_test dataset with 3651 rows and 21 columns saved to data/cleaned_test.csv
X_train dataset with 14603 rows and 21 columns saved to data/cleaned_train.csv


In [27]:
df = pd.read_csv('data/cleaned_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14603 entries, 0 to 14602
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unnamed: 0               14603 non-null  int64  
 1   idx                      14603 non-null  int64  
 2   Marca                    14603 non-null  object 
 3   Modelo                   14603 non-null  object 
 4   Año                      14603 non-null  float64
 5   Versión                  14090 non-null  object 
 6   Color                    14285 non-null  object 
 7   Tipo de combustible      14603 non-null  object 
 8   Puertas                  14603 non-null  float64
 9   Transmisión              14592 non-null  object 
 10  Motor                    14420 non-null  float64
 11  Tipo de carrocería       14603 non-null  object 
 12  Con cámara de retroceso  3738 non-null   object 
 13  Kilómetros               14603 non-null  float64
 14  Título                

In [29]:
# 1. Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Tipo de carrocería", "Título", "Descripción", "Precio", "Unnamed: 0"])

# 2. Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# 4. Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# 5. Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# 6. Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# 7. Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# 8. One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols)
X_val = pd.get_dummies(X_val, columns=one_hot_cols)

# 9. Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

# ✅ Ya tenés tus datasets limpios
print("Train:", X_train.shape)
print("Val:", X_val.shape)


Train: (11682, 38)
Val: (2921, 38)
